# Week 3: Exploratory Data Analysis & Preprocessing
## Notebook 07: Automated Data Profiling & Quality Health Checks from Scratch

### 1. Conceptual Intuition & Real-World Use Case
In production data engineering and automated ML pipelines:
- Manually writing `df.describe()` and checking plots for 100+ columns is slow and unscalable.
- Bad data silently poisons downstream machine learning models: unexpected nulls, zero variance (constant columns), skewed distributions, and high cardinality.

An **Automated Data Profiler** audits datasets programmatically, producing:
1. Missingness severity ratings.
2. Skewness and kurtosis anomaly alerts.
3. Near-zero variance column identification (candidates for removal).
4. High-cardinality flags on categorical features.

---

### 2. Mathematical Foundations: Distribution Metrics
1. **Skewness ($G_1$):**
   $$G_1 = \frac{n}{(n-1)(n-2)} \sum_{i=1}^n \left( \frac{x_i - \bar{x}}{s} \right)^3$$
   - $|G_1| > 1.0$: Highly skewed distribution (requires Log or Box-Cox transform).

2. **Excess Kurtosis ($G_2$):**
   $$G_2 = \frac{n(n+1)}{(n-1)(n-2)(n-3)} \sum_{i=1}^n \left( \frac{x_i - \bar{x}}{s} \right)^4 - \frac{3(n-1)^2}{(n-2)(n-3)}$$
   - $G_2 > 3.0$: Heavy tails (Leptokurtic - prone to extreme outliers).

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats

# Step 1: Create Synthetic Real-World Dataset with Data Quality Defects
np.random.seed(42)
N = 1000

df = pd.DataFrame({
    "user_id": np.arange(N),                                         # High cardinality ID
    "age": np.random.normal(35, 10, N),                              # Normal distribution
    "income": np.random.exponential(scale=50000, size=N),            # Severe positive skew
    "constant_flag": np.ones(N),                                     # Zero variance (defect!)
    "satisfaction": np.random.choice([1, 2, 3, 4, 5, np.nan], N, p=[0.1, 0.2, 0.3, 0.2, 0.1, 0.1]), # 10% nulls
    "device": np.random.choice(["iOS", "Android", "Web"], N)
})

# Step 2: Implement Automated Data Quality Audit Engine
class DataProfiler:
    def __init__(self, df):
        self.df = df
        self.total_rows = len(df)

    def audit(self):
        report = []
        for col in self.df.columns:
            series = self.df[col]
            null_count = series.isnull().sum()
            null_pct = (null_count / self.total_rows) * 100
            n_unique = series.nunique()
            unique_pct = (n_unique / self.total_rows) * 100
            
            alerts = []
            
            # Check 1: Missing Data Alert
            if null_pct > 0:
                alerts.append(f"Missing {null_pct:.1f}%")
            if null_pct > 30:
                alerts.append("CRITICAL_MISSING")

            # Check 2: Zero / Near-Zero Variance
            if n_unique <= 1:
                alerts.append("ZERO_VARIANCE")
            elif unique_pct > 90 and pd.api.types.is_numeric_dtype(series):
                alerts.append("POTENTIAL_ID_OR_KEY")

            # Check 3: Numeric Distribution Checks
            skewness = None
            kurt = None
            if pd.api.types.is_numeric_dtype(series) and n_unique > 1:
                clean_series = series.dropna()
                skewness = clean_series.skew()
                kurt = clean_series.kurtosis()
                
                if abs(skewness) > 1.5:
                    alerts.append(f"HIGH_SKEW ({skewness:.2f})")
                if kurt > 3.0:
                    alerts.append(f"HEAVY_TAILS (Kurt: {kurt:.2f})")

            report.append({
                "column": col,
                "dtype": str(series.dtype),
                "null_pct": f"{null_pct:.1f}%",
                "unique_count": n_unique,
                "skewness": round(skewness, 2) if skewness is not None else "N/A",
                "alerts": ", ".join(alerts) if alerts else "HEALTHY"
            })
            
        return pd.DataFrame(report)

profiler = DataProfiler(df)
audit_results = profiler.audit()

print("--- Automated Data Health Audit Report ---")
print(audit_results.to_string(index=False))

### 3. Step-by-Step Code Walkthrough
1. **Automated Defect Detection:** The profiler correctly flagged:
   - `constant_flag` as `ZERO_VARIANCE` (candidate for immediate deletion).
   - `income` as `HIGH_SKEW` and `HEAVY_TAILS` (signals log transform needed).
   - `user_id` as `POTENTIAL_ID_OR_KEY`.
   - `satisfaction` as `Missing 10.3%`.
2. **Programmatic Pipeline Integration:** In production, this dataframe can trigger automated alerts or halt CI/CD data ingestion pipelines if quality thresholds fail.

---

### 4. Industry Applications & Trade-offs
- **Great Expectations & Evidently AI:** Industry tools for monitoring data drift and silent pipeline degradation in production.
- **Trade-off:** Profiling massive multi-gigabyte tables requires sampling; running on 100M+ rows can increase pipeline execution time.

---

### 5. Top Interview Questions & Answers
**Q1: Why should zero-variance and near-zero-variance features be dropped before model training?**
> **Answer:** A feature with constant value provides zero discriminative information to split decision tree nodes or update linear model weights. Retaining it increases memory usage, adds noise, and in linear models causes singular matrix inversion issues in Ordinary Least Squares.

**Q2: What is the difference between positive skew and negative skew?**
> **Answer:** Positive skew (right-tailed) means the mean is greater than the median, with a long tail extending toward high positive values (e.g. income, housing prices). Negative skew (left-tailed) means the mean is less than the median, with a long tail extending toward low values (e.g. human age at retirement).